# Factor stability of the Cell-Cell Communication analysis

## Introduction

Here we test factor stability by re-running NMF decomposition.

## Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import liana as li

## Load data

In [2]:
lr_dir = os.path.join('out', 'lr')
lrdatas = {}

for fname in os.listdir(lr_dir):
    if fname.endswith('.h5ad'):
        cond_tissue = fname.replace('.h5ad', '')
        lrdatas[cond_tissue] = sc.read_h5ad(os.path.join(lr_dir, fname))

print(lrdatas[list(lrdatas.keys())[0]].var.head())

                ligand receptor  ligand_means  ligand_props  receptor_means  \
interaction                                                                   
ADAM10^TSPAN12  ADAM10  TSPAN12      0.344049      0.241343        0.147351   
ADAM10^CD44     ADAM10     CD44      0.344049      0.241343        0.499289   
ADAM10^NOTCH2   ADAM10   NOTCH2      0.344049      0.241343        0.136749   
HLA-B^LILRB2     HLA-B   LILRB2      4.015321      0.948855        0.232195   
HLA-B^LILRB1     HLA-B   LILRB1      4.015321      0.948855        0.278112   

                receptor_props      mean       std  
interaction                                         
ADAM10^TSPAN12        0.114544  0.159679  0.191081  
ADAM10^CD44           0.324454  0.247076  0.204881  
ADAM10^NOTCH2         0.108151  0.135250  0.166691  
HLA-B^LILRB2          0.182738  0.336642  0.228056  
HLA-B^LILRB1          0.206180  0.368736  0.233288  


In [3]:
lrdata_all = ad.concat(lrdatas, label = 'cond_tissue', join = 'outer')
print(lrdata_all)

AnnData object with n_obs × n_vars = 44852 × 1770
    obs: 'in_tissue', 'array_row', 'array_col', 'sample_id', 'cond_tissue', 'donor_tissue', 'study_id', 'disease_group', 'sex', 'age', 'sars.cov.2_antemortem_swap', 'sars.cov.2_postmortem_tissue', 'sars.cov.2_subgenomicRNA', 'nucleocapsid_IHC', 'tissue', 'lung_fibrosis_score', 'slide_id', 'capture_area', 'staining', 'sample_number', 'tcr_id', 'sum', 'detected', 'subsets_mito_sum', 'subsets_mito_detected', 'subsets_mito_percent', 'subsets_ribo_sum', 'subsets_ribo_detected', 'subsets_ribo_percent', 'subsets_SCoV2_sum', 'subsets_SCoV2_detected', 'subsets_SCoV2_percent', 'total', 'low_lib_size', 'low_n_features', 'discard', 'sizeFactor', 'lisi_study_id', 'annotation'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'spatial'


## LR pairs factorization with NMF with varying seed

NMF is sensitive to its random initialization, meaning different runs can converge to distinct local optima and yield factors that vary in composition and number. To assess whether the cell-cell communication (CCC) patterns identified by NMF are robust rather than artifacts of a particular initialization, we re-run the factorization 10 times using fixed, sequential random seeds (1 through 10) while keeping all other parameters fixed; fixed seeds are used instead of randomly generated ones to ensure the analysis is fully reproducible. We then compare the resulting factors across runs to evaluate consistency in the number of components selected, the ligand-receptor loadings, and the spatial/sample-level factor scores. Stable factors that recur across multiple random initializations are taken as more reliable representations of the underlying communication patterns, whereas factors that appear in only a subset of runs may reflect noise or instability in the decomposition.

In [4]:
features = pd.concat([lrdatas[cond_tissue].var for cond_tissue in lrdatas])
features['count'] = 1
features = features.reset_index().groupby('interaction').sum().sort_values('count')
print(f"LR pairs: {len(features)}")

features = features[features['count'] >= 3]
print(f"LR pairs retained: {len(features)}")
lrdata_all = sc.concat(lrdatas, join = 'outer', fill_value = 0)
lrdata_all = lrdata_all[:, features.index]
lrdata_all

LR pairs: 1770
LR pairs retained: 1283


View of AnnData object with n_obs × n_vars = 44852 × 1283
    obs: 'in_tissue', 'array_row', 'array_col', 'sample_id', 'cond_tissue', 'donor_tissue', 'study_id', 'disease_group', 'sex', 'age', 'sars.cov.2_antemortem_swap', 'sars.cov.2_postmortem_tissue', 'sars.cov.2_subgenomicRNA', 'nucleocapsid_IHC', 'tissue', 'lung_fibrosis_score', 'slide_id', 'capture_area', 'staining', 'sample_number', 'tcr_id', 'sum', 'detected', 'subsets_mito_sum', 'subsets_mito_detected', 'subsets_mito_percent', 'subsets_ribo_sum', 'subsets_ribo_detected', 'subsets_ribo_percent', 'subsets_SCoV2_sum', 'subsets_SCoV2_detected', 'subsets_SCoV2_percent', 'total', 'low_lib_size', 'low_n_features', 'discard', 'sizeFactor', 'lisi_study_id', 'annotation'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'spatial'

In [ ]:
nmf_results = {}

for seed in [*range(1, 11), 42]:
    print(f"Running NMF with random_state={seed}")
    lrdata_copy = lrdata_all.copy()
    
    li.multi.nmf(lrdata_copy,
                 n_components=None,
                 inplace=True,
                 max_iter=400,
                 k_range=range(1, 21),
                 verbose=True,
                 random_state=seed)
    
    for col in lrdata_copy.obs.columns:
        if lrdata_copy.obs[col].dtype == object:
            try:
                lrdata_copy.obs[col] = pd.to_numeric(lrdata_copy.obs[col], errors='raise')
            except (ValueError, TypeError):
                lrdata_copy.obs[col] = lrdata_copy.obs[col].astype(str)
    
    lrdata_copy.write_h5ad(os.path.join('out', 'nmf', f'seed{seed}.h5ad'))
    nmf_results[seed] = lrdata_copy

print(list(nmf_results.keys()))

Running NMF with random_state=1


100%|███████████████████████████████████████████| 20/20 [19:08<00:00, 57.43s/it]
Estimated rank: 5
/home/groups/singlecell/cvicente/miniconda3/envs/COVID_CCC/lib/python3.11/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 400 reached. Increase it to improve convergence.
... storing 'sars.cov.2_postmortem_tissue' as categorical
... storing 'nucleocapsid_IHC' as categorical
... storing 'lung_fibrosis_score' as categorical
... storing 'tcr_id' as categorical


Running NMF with random_state=2


 10%|████▍                                       | 2/20 [00:40<05:50, 19.48s/it]

## Session Information

In [2]:
import session_info
session_info.show()

/home/groups/singlecell/cvicente/miniconda3/envs/COVID_CCC/lib/python3.11/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
/home/groups/singlecell/cvicente/miniconda3/envs/COVID_CCC/lib/python3.11/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
